# Demo

## How to Run This Demo

### Important: Use a GPU Runtime for Best Performance
The BioM-BERT model is large (1.2GB). Running on **GPU is strongly recommended**:
- **GPU (e.g., Colab free T4):** 3 minutes
- **CPU:** 20 minutes

In Google Colab, go to **Runtime and change runtime type to GPU** before running.

### Steps
1. Upload the following 4 files to your Colab session (or place them in the same folder as this notebook):
   - `tfidf_augmented_15.pkl`
   - `CNN_LSTM_augmented_mtsample.pkl`
   - `exp6_aug15_gss.pkl`
   - `augmented_mtsamples.csv`
2. Run all cells (**Runtime -> Run all**).
3. Use the buttons in the **last cell** to view model results and confusion matrices.

> **Note:** The BioM-BERT button will take 3 minutes on GPU. A progress log will appear below the buttons while it loads.


## Imports

In [ ]:
#import statements
import sys
import pickle
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig
import ipywidgets as widgets
from IPython.display import display
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


## Loading Files

In [ ]:
# Loads the file for the saved TF-IDF model and makes sure all components are
# present
def load_tfidf_file():
  try:
    with open("tfidf_augmented_15.pkl", "rb") as f:
      file = pickle.load(f)
      vectorizer = file["vectorizer"]
      lr = file["lr"]
      svm = file["svm"]
      x_test = pd.Series(file["x_test"])
      y_test = pd.Series(file["y_test"])
      label_encoder = file["label_encoder"]
      return vectorizer, lr, svm, x_test, y_test, label_encoder
  except FileNotFoundError as e:
      print(f"There is an issue with the following file(s): {e}")
      print("Check to see if all files are uploaded and accessible.\n")
      return None, None, None, None, None, None

# Loads the file for the saved CNN-LSTM model and makes sure all components
# are present
def load_cnn_lstm_file():
  try:
      file = torch.load("CNN_LSTM_augmented_mtsample.pkl", map_location="cpu", weights_only=False)
      model = file["model"]
      vocab = file["vocab"]
      x_test = file["x_test"]
      y_test = file["y_test"]
      label_encoder = file["label_encoder"]
      model.eval()
      return model, vocab, x_test, y_test, label_encoder
  except FileNotFoundError as e:
      print(f"There is an issue with the following file(s): {e}")
      print("Check to see if all files are uploaded and accessible.\n")
      return None, None, None, None

# Loads the file for the saved BioM BERT model and makes sure all components
# are present
def load_biom_bert_file():
  try:
    with open("exp6_aug15_gss.pkl", "rb") as f:
      file = pickle.load(f)
      print("pkl loaded...")
      model = AutoModelForSequenceClassification.from_config(
          AutoConfig.from_pretrained(file["model_name"], num_labels=file["num_classes"])
      )
      print("model architecture built...")
      model.load_state_dict(file["model_state_dict"])
      print("weights loaded...")
      model.to(device)
      model.eval()
      tokenizer = AutoTokenizer.from_pretrained(file["model_name"])
      print("tokenizer loaded...")
      label_encoder = file["label_encoder"]
      max_length = file["max_length"]
      front = file["word_limit_front"]
      back = file["word_limit_back"]
      return model, tokenizer, label_encoder, max_length, front, back
  except FileNotFoundError as e:
    print(f"There is an issue with the following file(s): {e}")
    print("Check to see if all files are uploaded and accessible.\n")
    return None, None, None, None, None, None

## Display Helper Functions

In [ ]:
# Calculates the results. Takes in the y true and predicted values and returns
# the accuracy, macro precision, macro recall, weighted f1, and macro f1
def calc_results(y_true, y_pred):
  return { "accuracy": accuracy_score(y_true, y_pred),
            "macro_precision":precision_score(y_true, y_pred, average="macro", zero_division=0),
            "macro_recall": recall_score(y_true, y_pred, average="macro", zero_division=0),
            "weighted_f1": f1_score(y_true, y_pred, average="weighted", zero_division=0),
            "macro_f1" : f1_score(y_true, y_pred, average= "macro", zero_division=0)
          }

# Prints out the results. Takes in the results and prints it out.
def print_results(result):
  print(f"Accuracy: {result['accuracy']:.4f}")
  print(f"Macro Precision: {result['macro_precision']:.4f}")
  print(f"Macro Recall: {result['macro_recall']:.4f}")
  print(f"Weighted F1 Score: {result['weighted_f1']:.4f}")
  print(f"Macro F1 Score: {result['macro_f1']:.4f}")

# A consolidated confusion matrix plotting function that prints out a classification
# report and a confusion matrix. Takes in y true predicted labels, a label encoder,
# and a title
def plot_confusion_matrix(y_true, y_pred, label_encoder, title):
    numeric_labels = sorted(y_true.unique())
    # cnn-lstm's label_encoder for aug 15 is a list
    if isinstance(label_encoder, list):
      string_labels = [label_encoder[i] for i in numeric_labels]
    # for tfidf and biom bert
    else:
      string_labels = [label_encoder.classes_[i] for i in numeric_labels]

    print(f"\n{title}:")
    conf_mat = confusion_matrix(y_true, y_pred, labels=numeric_labels, normalize = 'true')
    plt.figure(figsize=(15, 10))
    sns.heatmap(conf_mat,
                annot=True,
                cmap="Blues",
                fmt=".2f",
                xticklabels=string_labels,
                yticklabels=string_labels)
    plt.title(title)
    plt.ylabel('Actual')
    plt.xlabel('Predicted')
    plt.show()

##TF-IDF Functions

In [ ]:
# functions that allow the information to be displayed

# Calculates and shows the TF-IDF Logistic Regression model results and confusion
# matrix.
def tfidf_lr_results():
  vectorizer, lr, svm, x_test, y_test,label_encoder = load_tfidf_file()
  print("\nTF-IDF Logistic Regression Results:")
  x_test_vec = vectorizer.transform(x_test)
  y_pred = lr.predict(x_test_vec)
  print_results(calc_results(y_test, y_pred))
  plot_confusion_matrix(y_test, y_pred, label_encoder, "LR Confusion Matrix")
  plt.close()

# Calculates and shows the TF-IDF LinearSVC/SVM model results and confusion matrix.
def tfidf_svm_results():
  vectorizer, lr, svm, x_test, y_test,label_encoder = load_tfidf_file()
  print("\nTF-IDF LinearSVC (SVM) Results:")
  x_test_vec = vectorizer.transform(x_test)
  y_pred = svm.predict(x_test_vec)
  print_results(calc_results(y_test, y_pred))
  plot_confusion_matrix(y_test, y_pred, label_encoder, "SVM Confusion Matrix")
  plt.close()

##CNN-LSTM Functions

In [ ]:
# copy paste class from cnn lstm notebook

class CNNLSTMModel(nn.Module):
    """
    Layer 1 : Embedding (Word2Vec initialization)
    Layer 2 : CNN  — Conv1d + Max Pooling
    Layer 3 : LSTM
    Layer 4 : Softmax
    """

    def __init__(self,
                 vocab_size: int,
                 embed_dim: int = 200,
                 num_filters: int = 128,
                 kernel_sizes: list[int] = None,
                 lstm_hidden: int = 200,
                 num_classes: int = 12,
                 dropout: float = 0.5,
                 pretrained_embeddings: np.ndarray = None):
        super().__init__()

        if kernel_sizes is None:
            kernel_sizes = [2, 3, 4]

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        if pretrained_embeddings is not None:
            self.embedding.weight.data.copy_(
                torch.from_numpy(pretrained_embeddings))

        self.convs = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(embed_dim, num_filters,
                          kernel_size=k, padding=k // 2),
                nn.ReLU()
            ) for k in kernel_sizes
        ])
        cnn_out_dim = num_filters * len(kernel_sizes)

        self.lstm = nn.LSTM(input_size=cnn_out_dim,
                            hidden_size=lstm_hidden,
                            num_layers=1,
                            batch_first=True,
                            bidirectional=False)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_hidden, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (batch, seq_len)
        """
        # Embedding: (batch, seq_len, embed_dim)
        emb = self.embedding(x)

        # CNN requires (batch, embed_dim, seq_len)
        emb_t = emb.permute(0, 2, 1)

        # multi convs (different kernel sizes) + Max Pooling → (batch, num_filters) × len(kernel_sizes)
        pooled = []
        for conv in self.convs:
            c = conv(emb_t)                 # (batch, num_filters, seq_len')
            p = torch.max(c, dim=2).values         # (batch, num_filters)  Max Pooling
            pooled.append(p)

        # concate different kernel results (batch, cnn_out_dim)
        cnn_out = torch.cat(pooled, dim=1)

        # simplier method：put cnn_out series as length = 1 into LSTM
        lstm_in = cnn_out.unsqueeze(1)             # (batch, 1, cnn_out_dim)
        lstm_out, _ = self.lstm(lstm_in)            # (batch, 1, lstm_hidden)
        lstm_feat = lstm_out[:, -1, :]            # (batch, lstm_hidden)

        out = self.dropout(lstm_feat)
        logits = self.fc(out)                  # (batch, num_classes)
        return logits

In [ ]:
# Converts tokens to their mapped numerical ID. Takes in a list of strings (tokens),
# a dictionary of mapped vocab to ID. Sets the maximum sequence length at 100.
def encode_tokens(tokens, vocab, max_len=100):
  ids = [vocab.get(w, 1) for w in tokens]
  if len(ids) >= max_len:
    return ids[:max_len]
  return ids + [0] * (max_len - len(ids))

# Uses the provided CNN-LSTM model to compute predictions on a dataset. Takes in
# a model, a dictionary of mapped voab to ID, and test tokens and returns an array
# of predicted labels
def predict_cnn_lstm(model, vocab, x_test_tokens):
  model.eval()
  all_preds = []
  with torch.no_grad():
    for tokens in x_test_tokens:
      # convert to ids
      ids = encode_tokens(tokens, vocab)
      x = torch.tensor([ids], dtype=torch.long)
      raw = model(x)
      pred = torch.argmax(raw, dim=1).item()
      # adds the prediction
      all_preds.append(pred)
  return np.array(all_preds)

# Calculates and shows the CNN-LSTM results and confusion matrix.
def cnn_lstm_results():
  print("\nCNN-LSTM Results:")
  model, vocab, x_test, y_test, label_encoder = load_cnn_lstm_file()
  if model is None:
    return

  y_pred = predict_cnn_lstm(model, vocab, x_test)
  y_test_series = pd.Series(y_test)
  y_pred_series = pd.Series(y_pred)

  print_results(calc_results(y_test_series, y_pred_series))
  plot_confusion_matrix(y_test_series, y_pred_series,label_encoder, "CNN-LSTM Confusion Matrix")
  plt.close()

## BioM-BERT Functions

In [ ]:
# copy pasted from BioM BERT
def smart_truncate_words(text: str, total_limit: int = 350,
                          front: int = 175, back: int = 175) -> str:
    """
    Truncates long transcriptions by keeping first 'front' words
    and last 'back' words rather than blindly cutting at 512 tokens.
    Clinical notes often have IMPRESSION/FINAL DIAGNOSIS at the END —
    standard truncation discards this critical diagnostic information.
    """
    words = text.split()
    if len(words) <= total_limit:
        return ' '.join(words)
    return ' '.join(words[:front] + words[-back:])

In [ ]:
# Uses the the provided BioM BERT model to compute predictions on the dataset.
# Takes in the model, tokenizer, texts(string), the maximum token length, and the
# number of words in the front and back. Sets the batch size at 16 and returns
# an array of predicted labels
def predict_bert(model, tokenizer, texts, max_length, front, back, batch_size=16):
  all_preds = []
  model.eval()
  for i in range(0, len(texts), batch_size):
    batch_texts = [smart_truncate_words(t, front, back) for t in texts[i:i + batch_size]]
    encoding = tokenizer(batch_texts,
                         max_length=max_length,
                         padding=True,
                         truncation=True,
                         return_tensors="pt")
    encoding = {k: v.to(device) for k, v in encoding.items()}
    with torch.no_grad():
      raw = model(**encoding).logits
      preds = torch.argmax(raw, dim=1).cpu().numpy()
      all_preds.extend(preds)
  return np.array(all_preds)

# Calculates and shows the results of BioM BERT
def biom_bert_results():
    from sklearn.model_selection import GroupShuffleSplit

    model, tokenizer, label_encoder, max_length, front, back = load_biom_bert_file()
    if model is None:
        return

    # x_test/y_test not saved in pkl so reproduce from CSV
    df = pd.read_csv("augmented_mtsamples.csv")
    gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
    _, test_idx = next(gss.split(df, groups=df['transcription']))
    test_df = df.iloc[test_idx].reset_index(drop=True)
    x_test = test_df['transcription'].tolist()
    y_test = label_encoder.transform(test_df['medical_specialty'])

    y_pred = predict_bert(model, tokenizer, x_test, max_length, front, back)
    y_test_series = pd.Series(y_test)
    y_pred_series = pd.Series(y_pred)

    print_results(calc_results(y_test_series, y_pred_series))
    plot_confusion_matrix(y_test_series, y_pred_series, label_encoder, "BioM-BERT Confusion Matrix")

## Text Based Model Selection

In [7]:
# Shows a selection of models that the user can view the results of for the best
# performing dataset, Augmented 15 and waits for the user's input.
import ipywidgets as widgets
from IPython.display import display

def model_selection():
    print("Medical Text Classification Models (Aug-15):")

    out = widgets.Output()

    btn_tfidf_lr  = widgets.Button(description="TF-IDF LR Results")
    btn_tfidf_svm = widgets.Button(description="TF-IDF SVM Results")
    btn_cnn_lstm  = widgets.Button(description="CNN-LSTM Results")
    btn_biom_bert = widgets.Button(description="BioM-BERT Results")

    def on_tfidf_lr(b):
        with out:
            out.clear_output()
            print("Loading TF-IDF Logistic Regression results...")
            try:
                tfidf_lr_results()
            except Exception as e:
                print(f"Error: {e}")

    def on_tfidf_svm(b):
        with out:
            out.clear_output()
            print("Loading TF-IDF SVM results...")
            try:
                tfidf_svm_results()
            except Exception as e:
                print(f"Error: {e}")

    def on_cnn_lstm(b):
        with out:
            out.clear_output()
            print("Loading CNN-LSTM results...")
            try:
                cnn_lstm_results()
            except Exception as e:
                print(f"Error: {e}")

    def on_biom_bert(b):
        with out:
            out.clear_output()
            print("Loading BioM-BERT... please wait (GPU: 3 min, CPU: 20 min)")
            try:
                biom_bert_results()
            except Exception as e:
                print(f"Error: {e}")

    btn_tfidf_lr.on_click(on_tfidf_lr)
    btn_tfidf_svm.on_click(on_tfidf_svm)
    btn_cnn_lstm.on_click(on_cnn_lstm)
    btn_biom_bert.on_click(on_biom_bert)

    display(widgets.VBox([btn_tfidf_lr, btn_tfidf_svm, btn_cnn_lstm, btn_biom_bert, out]))

In [8]:
# Run the Demo by running this cell
model_selection()

Medical Text Classification Models (Aug-15):
